In [1]:
# === Core Python ===
import os
import glob
import re
import collections
import cftime
from datetime import datetime
from typing import Tuple, Dict, Optional, List
from pathlib import Path

# === Numerical & Data Handling ===
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xcd
import xskillscore as xs
from xskillscore import rmse, pearson_r

from dataclasses import dataclass

from DataUtil import (
    build_experiments,
    DEFAULT_EXPERIMENTS
)

from ObsUtil import (
    OBS_REGISTRY,
    get_obs_file,
    list_obs_sources,
    obs_coverage,
)

from time_window import(
    build_windows_from_start
)


In [3]:
@dataclass(frozen=True)
class RegionSpec:
    name: str
    lat: tuple[float, float]
    lon: tuple[float, float]
    mask: str = "none"   # "none" | "land" | "ocean"


# ----------------------------
# Config
# ----------------------------

@dataclass(frozen=True)
class RegionalReduceConfig:
    out_dir: Path
    overwrite: bool = False
    verbose: bool = False

    # ens naming (members)
    ens_start: int = 1
    ens_prefix: str = "EN"
    ens_width: int = 2

    # ensemble-mean (skill of ensemble-mean forecast field)
    include_ensmean: bool = True
    ensmean_tag: str = "ENSMEAN"
    compute_ensmean_if_missing: bool = False  # keep False for your case

    # coords
    lat_names: tuple[str, ...] = ("lat", "latitude", "nlat")
    lon_names: tuple[str, ...] = ("lon", "longitude", "nlon")

    # mask
    landmask_file: Path | None = None
    landmask_var: str = "landfrac"
    land_threshold: float = 0.5

    # quantiles for member distribution (e.g., for shading)
    member_quantiles: tuple[float, ...] = (0.10, 0.25, 0.50, 0.75, 0.90)

    # Optional: require at least this fraction of region weights to be valid; otherwise set mean to NaN
    min_valid_frac: float = 0.0  # set to e.g. 0.05 if desired

    # NEW: save both quantile definitions
    save_member_quantiles_post: bool = True  # quantile(regional_mean over ens)
    save_member_quantiles_pre: bool = True   # regional_mean(quantile map over ens)


# ----------------------------
# Reducer
# ----------------------------

class S2SRegionalSkillReducer:
    """
    Reads per-member base files (and optional ENSMEAN base file) containing spatial skill maps
    (e.g., TCC/RMSE) on the model grid and computes regional area-weighted means.

    ENSMEAN is treated as a separate product: "skill of ensemble-mean forecast field",
    and is NOT included in member ensemble statistics.

    Quantiles:
      - qpostXX: quantile across ens of the *regional-mean* member skill (current behavior)
      - qpreXX : regional-mean of the *gridpoint* ensemble quantile map (new behavior)

    Key robustness:
      - Canonicalizes longitude to 0..360 for BOTH data and landmask
      - Region selection uses wrap-aware logic and treats (0,360) as full globe
      - Handles NaNs by computing weighted mean only over finite values and
        outputs valid-weight sums + valid fractions (optionally masks low coverage)
      - Removes input 'region' dim/coord before creating output region dim
    """

    def __init__(self, cfg: RegionalReduceConfig, regions: tuple[RegionSpec, ...]):
        self.cfg = cfg
        self.regions = regions
        self.cfg.out_dir.mkdir(parents=True, exist_ok=True)

    # ---------- utilities ----------
    def _pick_coord(self, ds: xr.Dataset, names: tuple[str, ...], kind: str) -> str:
        for n in names:
            if n in ds.coords or n in ds.dims:
                return n
        raise KeyError(
            f"Cannot find {kind} coord/dim. Tried {names}. "
            f"Found coords={list(ds.coords)}, dims={list(ds.dims)}"
        )

    def _lon_to_0_360_1d(self, lon: xr.DataArray) -> xr.DataArray:
        """Convert 1D lon coordinate to [0, 360) and ensure increasing order."""
        lon2 = lon.copy()
        if float(lon2.min()) < 0.0:
            lon2 = lon2 % 360.0

        if lon2.ndim == 1:
            vals = lon2.values
            if not np.all(np.diff(vals) >= 0):
                order = np.argsort(vals)
                lon2 = lon2.isel({lon2.dims[0]: order})
        return lon2

    def _canonicalize_lon_ds(self, ds: xr.Dataset, lon_name: str) -> xr.Dataset:
        """Canonicalize ds lon coordinate to 0..360 and sort ds by lon if necessary. Assumes lon is 1D."""
        if lon_name not in ds.coords and lon_name not in ds.dims:
            return ds

        lon_old = ds[lon_name]
        lon_new = self._lon_to_0_360_1d(lon_old)
        ds2 = ds.assign_coords({lon_name: lon_new})
        ds2 = ds2.sortby(lon_name)
        return ds2

    def _normalize_region_lon_0_360(self, lon0: float, lon1: float) -> tuple[float, float, bool]:
        """Normalize region lon bounds to 0..360. Returns (lonmin, lonmax, is_full_globe)."""
        span = lon1 - lon0
        if span >= 360.0 - 1e-6:
            return 0.0, 360.0, True
        lonmin = lon0 % 360.0
        lonmax = lon1 % 360.0
        return lonmin, lonmax, False

    def _clean_input_region(self, ds: xr.Dataset) -> xr.Dataset:
        """
        Clean the input 'region' axis from base files:
          - if region dim exists and is singleton: squeeze it (also removes index coord)
          - if region dim exists and is non-singleton: rename to region_in to avoid collision
          - drop any leftover non-index coord/var named 'region'
        """
        if "region" in ds.dims:
            n = ds.sizes.get("region", 0)
            if n == 1:
                ds = ds.squeeze("region", drop=True)
            else:
                ds = ds.rename({"region": "region_in"})

        if "region" in ds.coords:
            try:
                ds = ds.reset_coords("region", drop=True)
            except ValueError:
                ds = ds.rename({"region": "region_in_coord"})

        if "region" in ds.variables and "region" not in ds.dims:
            ds = ds.drop_vars("region", errors="ignore")

        return ds

    def _sanitize_region_for_output(self, ds: xr.Dataset) -> xr.Dataset:
        """Ensure there is NO 'region' dim/coord/var before creating OUTPUT region dim."""
        if "region" in ds.dims:
            n = ds.sizes.get("region", 0)
            if n == 1:
                ds = ds.squeeze("region", drop=True)
            else:
                ds = ds.rename({"region": "region_in"})

        if "region" in ds.coords:
            ds = ds.reset_coords("region", drop=True)

        if "region" in ds.variables and "region" not in ds.dims:
            ds = ds.drop_vars("region", errors="ignore")

        return ds

    def _squeeze_region_da(self, da: xr.DataArray) -> xr.DataArray:
        if "region" in da.dims:
            n = da.sizes.get("region", 0)
            if n == 1:
                da = da.squeeze("region", drop=True)
            else:
                raise ValueError(f"Unexpected non-singleton 'region' dim: {da.sizes['region']}")
        return da

    def _area_w2d(self, lat: xr.DataArray, lon: xr.DataArray) -> xr.DataArray:
        """Area weight ~ cos(lat); broadcast to (lat, lon)."""
        w1 = np.cos(np.deg2rad(lat))
        w1 = xr.where(w1 < 0, 0.0, w1)
        return w1.broadcast_like(
            xr.DataArray(
                np.zeros((lat.size, lon.size)),
                coords={lat.name: lat, lon.name: lon},
                dims=(lat.name, lon.name),
            )
        )

    def _ens_tag(self, k: int) -> str:
        return f"{self.cfg.ens_prefix}{k:0{self.cfg.ens_width}d}"

    def _canonical_metric_name(self, vn: str) -> str:
        base = vn
        for suf in ("_ensmean", "_member"):
            if base.endswith(suf):
                base = base[: -len(suf)]
        return base

    def _validate_quantiles(self) -> tuple[float, ...]:
        qs = tuple(sorted(set(self.cfg.member_quantiles)))
        for q in qs:
            if not (0.0 <= q <= 1.0):
                raise ValueError(f"member_quantiles must be in [0,1]. Got {q}")
        return qs

    # ---------- file collection ----------
    def collect_member_and_ensmean_files(
        self,
        input_dir: Path,
        *,
        group: str,
        freq: str,
        run: str,
        obs: str,
        period: str,
        exp: str,
        var: str,
        nens: int,
    ) -> tuple[list[Path], list[int], Path | None]:
        member_files: list[Path] = []
        member_ens: list[int] = []

        for k in range(self.cfg.ens_start, self.cfg.ens_start + nens):
            tag = self._ens_tag(k)
            f = input_dir / f"s2s_base_tcc_rmse_{group}_{freq}_{run}_{obs}_{period}_ALL_{exp}_{tag}_{var}.nc"
            if f.exists():
                member_files.append(f)
                member_ens.append(k)

        ensmean_file = None
        if self.cfg.include_ensmean:
            tag = self.cfg.ensmean_tag
            f = input_dir / f"s2s_base_tcc_rmse_{group}_{freq}_{run}_{obs}_{period}_ALL_{exp}_{tag}_{var}.nc"
            if f.exists():
                ensmean_file = f

        return member_files, member_ens, ensmean_file

    # ---------- region + landmask ----------
    def _region_bool(self, lat: xr.DataArray, lon: xr.DataArray, reg: RegionSpec) -> xr.DataArray:
        """Region mask on canonical lon (expected 0..360). Wrap-aware; (0,360) treated as full globe."""
        lat2d, lon2d = xr.broadcast(lat, lon)

        latmin, latmax = reg.lat
        in_lat = (lat2d >= latmin) & (lat2d <= latmax)

        lon0, lon1 = reg.lon
        lonmin, lonmax, full = self._normalize_region_lon_0_360(lon0, lon1)

        if full:
            in_lon = xr.ones_like(lon2d, dtype=bool)
        else:
            if lonmin <= lonmax:
                in_lon = (lon2d >= lonmin) & (lon2d <= lonmax)
            else:
                in_lon = (lon2d >= lonmin) | (lon2d <= lonmax)

        return (in_lat & in_lon).transpose(lat.name, lon.name)

    def _load_landmask_bool(self, target_lat: xr.DataArray, target_lon: xr.DataArray, want: str) -> xr.DataArray:
        """want: 'none' | 'land' | 'ocean'"""
        if want == "none":
            return xr.ones_like(self._area_w2d(target_lat, target_lon), dtype=bool)

        if self.cfg.landmask_file is None:
            raise ValueError("Region requests land/ocean mask but cfg.landmask_file is None")

        ds_m = xr.open_dataset(self.cfg.landmask_file, decode_times=False)
        if self.cfg.landmask_var not in ds_m:
            raise KeyError(
                f"landmask_var={self.cfg.landmask_var} not in {self.cfg.landmask_file}. "
                f"Found {list(ds_m.data_vars)}"
            )

        latm = self._pick_coord(ds_m, self.cfg.lat_names, "lat")
        lonm = self._pick_coord(ds_m, self.cfg.lon_names, "lon")

        ds_m = self._canonicalize_lon_ds(ds_m, lonm)
        m = ds_m[self.cfg.landmask_var]

        target_lon = self._lon_to_0_360_1d(target_lon)

        if (m.sizes.get(latm) != target_lat.size) or (m.sizes.get(lonm) != target_lon.size):
            m = m.interp({latm: target_lat, lonm: target_lon}, method="nearest")

        is_land = (m >= self.cfg.land_threshold)
        out = is_land if want == "land" else (~is_land)

        out = out.rename({latm: target_lat.name, lonm: target_lon.name})
        return out.astype(bool).transpose(target_lat.name, target_lon.name)

    # ---------- math helpers ----------
    def _weighted_mean_latlon(self, da: xr.DataArray, w2d: xr.DataArray, lat_name: str, lon_name: str) -> xr.DataArray:
        """Weighted mean over lat/lon using weights w2d, ignoring NaNs in da."""
        w = w2d
        for d in da.dims:
            if d not in w.dims:
                w = w.expand_dims({d: da.sizes[d]})
        w = w.transpose(*da.dims)

        valid = xr.ufuncs.isfinite(da)
        w_eff = w.where(valid, 0.0)

        num = (da.where(valid) * w_eff).sum(dim=(lat_name, lon_name), skipna=True)
        den = w_eff.sum(dim=(lat_name, lon_name), skipna=True)

        out = num / den
        out = out.where(den > 0)
        return out

    def _valid_weight_sum(self, da: xr.DataArray, w2d: xr.DataArray, lat_name: str, lon_name: str) -> xr.DataArray:
        """Sum of weights in region where da is finite."""
        w = w2d
        for d in da.dims:
            if d not in w.dims:
                w = w.expand_dims({d: da.sizes[d]})
        w = w.transpose(*da.dims)

        valid = xr.ufuncs.isfinite(da)
        return w.where(valid, 0.0).sum(dim=(lat_name, lon_name), skipna=True)

    # ---------- open helpers ----------
    def _open_stack_members(self, member_files: list[Path], member_ens: list[int]) -> xr.Dataset:
        dsets = []
        for f, ensv in zip(member_files, member_ens):
            ds = xr.open_dataset(f)
            ds = self._clean_input_region(ds)
            ds = ds.expand_dims({"ens": [ensv]})
            dsets.append(ds)
        return xr.concat(dsets, dim="ens")

    def _open_ensmean(self, ensmean_file: Path) -> xr.Dataset:
        ds = xr.open_dataset(ensmean_file)
        ds = self._clean_input_region(ds)
        return ds

    # ---------- main API ----------
    def reduce_one_exp_var(
        self,
        input_dir: Path,
        *,
        group: str,
        freq: str,
        run: str,
        obs: str,
        period: str,
        exp: str,
        var: str,
        nens: int,
    ) -> Path | None:
        member_files, member_ens, ensmean_file = self.collect_member_and_ensmean_files(
            input_dir,
            group=group, freq=freq, run=run, obs=obs, period=period, exp=exp, var=var, nens=nens
        )

        if not member_files and ensmean_file is None:
            if self.cfg.verbose:
                print(f"[miss] {group} {obs} {exp} {var}: no member or ensmean files")
            return None

        return self.reduce_members_to_one_output(
            member_files=member_files,
            member_ens=member_ens,
            ensmean_file=ensmean_file,
            group=group, obs=obs, period=period, exp=exp, var=var,
        )

    def reduce_members_to_one_output(
        self,
        member_files: list[Path],
        member_ens: list[int],
        ensmean_file: Path | None,
        *,
        group: str,
        obs: str,
        period: str,
        exp: str,
        var: str,
    ) -> Path:
        out_name = f"s2s_regional_tcc_rmse_{group}_{obs}_{period}_{exp}_{var}.nc"
        out_path = self.cfg.out_dir / out_name

        if out_path.exists() and not self.cfg.overwrite:
            if self.cfg.verbose:
                print(f"[skip] exists: {out_path}")
            return out_path

        qs = self._validate_quantiles()

        ds_mem = self._open_stack_members(member_files, member_ens) if member_files else None
        ds_em = self._open_ensmean(ensmean_file) if ensmean_file is not None else None

        ds_ref = ds_mem if ds_mem is not None else ds_em
        if ds_ref is None:
            raise RuntimeError("Internal: no reference dataset available.")

        lat_name = self._pick_coord(ds_ref, self.cfg.lat_names, "lat")
        lon_name = self._pick_coord(ds_ref, self.cfg.lon_names, "lon")

        if ds_mem is not None:
            ds_mem = self._canonicalize_lon_ds(ds_mem, lon_name)
        if ds_em is not None:
            ds_em = self._canonicalize_lon_ds(ds_em, lon_name)

        ds_ref2 = ds_mem if ds_mem is not None else ds_em
        if ds_ref2 is None:
            raise RuntimeError("Internal: no reference dataset available after lon canonicalization.")

        lat = ds_ref2[lat_name]
        lon = ds_ref2[lon_name]

        # map vars that are maps (have lat/lon)
        map_vars_mem = []
        if ds_mem is not None:
            map_vars_mem = [vn for vn, da in ds_mem.data_vars.items() if (lat_name in da.dims and lon_name in da.dims)]
        map_vars_em = []
        if ds_em is not None:
            map_vars_em = [vn for vn, da in ds_em.data_vars.items() if (lat_name in da.dims and lon_name in da.dims)]

        mem_by_metric: dict[str, list[str]] = {}
        for vn in map_vars_mem:
            mem_by_metric.setdefault(self._canonical_metric_name(vn), []).append(vn)

        em_by_metric: dict[str, list[str]] = {}
        for vn in map_vars_em:
            em_by_metric.setdefault(self._canonical_metric_name(vn), []).append(vn)

        metrics = sorted(set(mem_by_metric.keys()) | set(em_by_metric.keys()))
        if self.cfg.verbose:
            print(f"[reduce] {group} {obs} {exp} {var}: metrics={metrics}")
            if ds_em is not None:
                print(f"         ensmean map vars: {map_vars_em}")

        w_area = self._area_w2d(lat, lon)

        lo_cache: dict[str, xr.DataArray] = {"none": xr.ones_like(w_area, dtype=bool)}
        if any(r.mask == "land" for r in self.regions):
            lo_cache["land"] = self._load_landmask_bool(lat, lon, "land")
        if any(r.mask == "ocean" for r in self.regions):
            lo_cache["ocean"] = self._load_landmask_bool(lat, lon, "ocean")

        out_list: list[xr.Dataset] = []
        for reg in self.regions:
            m_reg = self._region_bool(lat, lon, reg)
            m_lo = lo_cache.get(reg.mask)
            if m_lo is None:
                m_lo = self._load_landmask_bool(lat, lon, reg.mask)
            m = (m_reg & m_lo)

            w = w_area.where(m, 0.0)

            reg_ds = xr.Dataset()

            region_wsum = w.sum(dim=(lat_name, lon_name), skipna=True)
            region_wsum = self._squeeze_region_da(region_wsum)
            reg_ds["region_area_weight_sum"] = region_wsum

            if ds_mem is not None:
                for metric in metrics:
                    vns = mem_by_metric.get(metric, [])
                    if not vns:
                        continue
                    vn = vns[0]  # assume one map var per metric in member dataset

                    reg_mean_mem = self._weighted_mean_latlon(ds_mem[vn], w, lat_name, lon_name)
                    reg_mean_mem = self._squeeze_region_da(reg_mean_mem)

                    vw_mem = self._valid_weight_sum(ds_mem[vn], w, lat_name, lon_name)
                    vw_mem = self._squeeze_region_da(vw_mem)

                    valid_frac_mem = (vw_mem / region_wsum).where(region_wsum > 0)

                    if self.cfg.min_valid_frac > 0:
                        reg_mean_mem = reg_mean_mem.where(valid_frac_mem >= self.cfg.min_valid_frac)

                    reg_ds[f"{metric}_regional_member"] = reg_mean_mem
                    reg_ds[f"{metric}_valid_wsum_member"] = vw_mem
                    reg_ds[f"{metric}_valid_frac_member"] = valid_frac_mem

                    reg_ds[f"{metric}_member_ens_mean"] = reg_mean_mem.mean("ens", skipna=True)
                    reg_ds[f"{metric}_member_ens_std"] = reg_mean_mem.std("ens", skipna=True)

                    # ---- Quantiles (POST): quantile of regional mean across members ----
                    if self.cfg.save_member_quantiles_post:
                        with np.errstate(all="ignore"):
                            for q in qs:
                                qlab = int(round(q * 100))
                                reg_ds[f"{metric}_member_qpost{qlab:02d}"] = reg_mean_mem.quantile(
                                    q, dim="ens", skipna=True
                                )

                    # ---- Quantiles (PRE): regional mean of gridpoint quantile maps ----
                    if self.cfg.save_member_quantiles_pre:
                        # qmap dims: (q, ... , lat, lon)  (q coordinate name: "q")
                        qmap = ds_mem[vn].quantile(qs, dim="ens", skipna=True).rename({"quantile": "q"})
                        qreg = self._weighted_mean_latlon(qmap, w, lat_name, lon_name)
                        qreg = self._squeeze_region_da(qreg)

                        with np.errstate(all="ignore"):
                            for q in qs:
                                qlab = int(round(q * 100))
                                reg_ds[f"{metric}_member_qpre{qlab:02d}"] = qreg.sel(q=q).drop_vars("q", errors="ignore")

            if ds_em is not None:
                for metric in metrics:
                    vns = em_by_metric.get(metric, [])
                    if not vns:
                        continue
                    vn = vns[0]  # assume one map var per metric in ensmean dataset

                    reg_mean_em = self._weighted_mean_latlon(ds_em[vn], w, lat_name, lon_name)
                    reg_mean_em = self._squeeze_region_da(reg_mean_em)

                    vw_em = self._valid_weight_sum(ds_em[vn], w, lat_name, lon_name)
                    vw_em = self._squeeze_region_da(vw_em)

                    valid_frac_em = (vw_em / region_wsum).where(region_wsum > 0)

                    if self.cfg.min_valid_frac > 0:
                        reg_mean_em = reg_mean_em.where(valid_frac_em >= self.cfg.min_valid_frac)

                    reg_ds[f"{metric}_regional_ensmean"] = reg_mean_em
                    reg_ds[f"{metric}_valid_wsum_ensmean"] = vw_em
                    reg_ds[f"{metric}_valid_frac_ensmean"] = valid_frac_em

            reg_ds = self._sanitize_region_for_output(reg_ds)
            reg_ds = reg_ds.expand_dims({"region": [reg.name]})
            reg_ds = reg_ds.assign_attrs(region_mask=reg.mask)
            out_list.append(reg_ds)

        ds_out = xr.concat(out_list, dim="region")
        ds_out = ds_out.assign_attrs(
            group=group,
            obs=obs,
            exp=exp,
            var=var,
            period=period,
            member_quantiles=",".join(f"{q:.3f}" for q in qs),
            min_valid_frac=str(self.cfg.min_valid_frac),
            source_member_files=";".join(str(p) for p in member_files) if member_files else "",
            source_ensmean_file=str(ensmean_file) if ensmean_file is not None else "",
            member_quantile_post_definition="qpostXX = quantile over ens of regional-mean member skill",
            member_quantile_pre_definition="qpreXX = regional-mean of gridpoint quantile map over ens",
        )

        ds_out.to_netcdf(out_path)
        return out_path

ERROR! Session/line number was not unique in database. History logging moved to new session 23


In [ ]:
if __name__ == "__main__":
    data_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    diag_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base"
    land_mask = "/compyfs/zhan391/acme_init/lnd_sea_mask/landmask_1x1.nc"

    # build exp metadata once
    exp_dict = build_experiments(data_path)

    # landmask settings
    landmask_var = "landfrac"
    land_threshold = 0.5

    exp_list = {
        "Jan2012": dict(
            models=["CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"],
            period="201201-201203",
            season="Winter",
            init_date="2012-01-01",
            init_month=1,
            run="fc",
            freq="daily",
        ),
        "Jun2012": dict(
            models=["CTRL10-S1", "CAPT10-S1", "DART40-S1"],
            period="201206-201208",
            season="Summer",
            init_date="2012-06-01",
            init_month=6,
            run="fc",
            freq="daily",
        ),
    }

    # obs -> {model_var -> obs_var}
    s2s_var_dict = {
        "ERA5": {
            "UBOT": "U10",
            "VBOT": "V10",
            "PRECT": "PRECT",
            "PSL": "PSL",
            "U850": "U850",
            "V850": "V850",
            "T850": "T850",
            "Q850": "Q850",
            "Z500": "Z500",
            "OMEGA500": "OMEGA500",
            "OMEGA200": "OMEGA200",
            "OMEGA850": "OMEGA850",
            "U200": "U200",
            "V200": "V200",
            "U500": "U500",
            "V500": "V500",
            "FLDS": "FLDS",
            "FLDSC": "FLDSC",
            "FLNS": "FLNS",
            "FLNSC": "FLNSC",
            "FSDS": "FSDS",
            "FSDSC": "FSDSC",
            "FSNS": "FSNS",
            "FSNSC": "FSNSC",
            "LHFLX": "LHFLX",
            "SHFLX": "SHFLX",
            "QFLX": "QFLX",
            "TREFHT": "TREFHT",
            "TS": "TS",
            "PRECSL": "PRECSL",
            "PRECC": "PRECC",
            "PRECL": "PRECL",
        },
        "GPCP": {"PRECT": "PRECT"},
        "GPM": {"PRECT": "PRECT"},
        "NOAA-OLR": {"FLUT": "FLUT"},
    }

    region_dict = [
        dict(name="GLOBAL", lat=(-90.0, 90.0), lon=(0.0, 360.0)),
        dict(name="OCEAN", lat=(-90.0, 90.0), lon=(0.0, 360.0), mask="ocean"),
        dict(name="LAND", lat=(-90.0, 90.0), lon=(0.0, 360.0), mask="land"),
        dict(name="NINO3.4", lat=(-5.0, 5.0), lon=(190.0, 240.0)),
        dict(name="MAR_CONT", lat=(-10.0, 10.0), lon=(95.0, 150.0)),
        dict(name="CONUS", lat=(25.0, 50.0), lon=(235.0, 294.0)),
        dict(name="NH_POLAR", lat=(60.0, 90.0), lon=(0.0, 360.0)),
    ]

    regions = tuple(
        RegionSpec(
            name=r["name"],
            lat=r["lat"],
            lon=r["lon"],
            mask=r.get("mask", "none"),
        )
        for r in region_dict
    )

    # config knobs
    ens_start = 1
    ens_prefix = "EN"
    ens_width = 2
    overwrite = True
    verbose = True

    member_quantiles = (0.05, 0.25, 0.50, 0.75, 0.90, 0.95)
    include_ensmean = True
    ensmean_tag = "ENSMEAN"
    compute_ensmean_if_missing = False

    output_dir = Path(f"{diag_path}/regional")
    output_dir.mkdir(parents=True, exist_ok=True)

    land_mask_path = Path(land_mask)
    if any(r.get("mask", "none") in ("land", "ocean") for r in region_dict) and (not land_mask_path.exists()):
        raise FileNotFoundError(f"land_mask not found: {land_mask_path}")

    cfg = RegionalReduceConfig(
        out_dir=output_dir,
        overwrite=overwrite,
        verbose=verbose,
        ens_start=ens_start,
        ens_prefix=ens_prefix,
        ens_width=ens_width,
        include_ensmean=include_ensmean,
        ensmean_tag=ensmean_tag,
        compute_ensmean_if_missing=compute_ensmean_if_missing,
        member_quantiles=member_quantiles,
        landmask_file=land_mask_path,
        landmask_var=landmask_var,
        land_threshold=land_threshold,
    )
    reducer = S2SRegionalSkillReducer(cfg, regions=regions)

    groups = list(exp_list.keys())
    for group in groups:
        input_dir = Path(f"{diag_path}/{group}")
        if not input_dir.exists():
            raise FileNotFoundError(f"input_dir not found: {input_dir}")

        freq = exp_list[group]["freq"]
        run = exp_list[group]["run"]
        models = exp_list[group]["models"]
        period = exp_list[group]["period"]

        for obs, var_map in s2s_var_dict.items():
            for exp in models:
                nens = int(exp_dict[exp]["nens"])
                for var in var_map.keys():
                    if verbose:
                        print(f"[run] {group} {freq} {run} {obs} {period} {exp} {var}")

                    out_nc = reducer.reduce_one_exp_var(
                        input_dir,
                        group=group,
                        freq=freq,
                        run=run,
                        obs=obs,
                        period=period,
                        exp=exp,
                        var=var,
                        nens=nens,
                    )
                    if out_nc is not None:
                        print(out_nc)

[run] Jan2012 daily fc ERA5 201201-201203 CTRL10-S0 UBOT
[reduce] Jan2012 ERA5 CTRL10-S0 UBOT: metrics=['RMSE', 'TCC']
         ensmean map vars: ['TCC_ensmean', 'RMSE_ensmean']
/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/regional/s2s_regional_tcc_rmse_Jan2012_ERA5_201201-201203_CTRL10-S0_UBOT.nc
[run] Jan2012 daily fc ERA5 201201-201203 CTRL10-S0 VBOT
[reduce] Jan2012 ERA5 CTRL10-S0 VBOT: metrics=['RMSE', 'TCC']
         ensmean map vars: ['TCC_ensmean', 'RMSE_ensmean']
/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/regional/s2s_regional_tcc_rmse_Jan2012_ERA5_201201-201203_CTRL10-S0_VBOT.nc
[run] Jan2012 daily fc ERA5 201201-201203 CTRL10-S0 PRECT
[reduce] Jan2012 ERA5 CTRL10-S0 PRECT: metrics=['RMSE', 'TCC']
         ensmean map vars: ['TCC_ensmean', 'RMSE_ensmean']


/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/

/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/regional/s2s_regional_tcc_rmse_Jan2012_ERA5_201201-201203_CTRL10-S0_PRECT.nc
[run] Jan2012 daily fc ERA5 201201-201203 CTRL10-S0 PSL
[reduce] Jan2012 ERA5 CTRL10-S0 PSL: metrics=['RMSE', 'TCC']
         ensmean map vars: ['TCC_ensmean', 'RMSE_ensmean']
/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/regional/s2s_regional_tcc_rmse_Jan2012_ERA5_201201-201203_CTRL10-S0_PSL.nc
[run] Jan2012 daily fc ERA5 201201-201203 CTRL10-S0 U850
[reduce] Jan2012 ERA5 CTRL10-S0 U850: metrics=['RMSE', 'TCC']
         ensmean map vars: ['TCC_ensmean', 'RMSE_ensmean']
/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/regional/s2s_regional_tcc_rmse_Jan2012_ERA5_201201-201203_CTRL10-S0_U850.nc
[run] Jan2012 daily fc ERA5 201201-201203 CTRL10-S0 V850
[reduce] Jan2012 ERA5 CTRL10-S0 V850: metrics=['RMSE', 'TCC']
         ensmean map vars: ['TCC_ensmean', 'RMSE_ensmean']
/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tc

/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/

/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/regional/s2s_regional_tcc_rmse_Jan2012_ERA5_201201-201203_CTRL10-S0_FLNS.nc
[run] Jan2012 daily fc ERA5 201201-201203 CTRL10-S0 FLNSC
[reduce] Jan2012 ERA5 CTRL10-S0 FLNSC: metrics=['RMSE', 'TCC']
         ensmean map vars: ['TCC_ensmean', 'RMSE_ensmean']


...

/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/

/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/regional/s2s_regional_tcc_rmse_Jan2012_GPM_201201-201203_CAPT10-S0_PRECT.nc
[run] Jan2012 daily fc GPM 201201-201203 DART20-S0 PRECT
[reduce] Jan2012 GPM DART20-S0 PRECT: metrics=['RMSE', 'TCC']
         ensmean map vars: ['TCC_ensmean', 'RMSE_ensmean']


/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/

/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/regional/s2s_regional_tcc_rmse_Jan2012_GPM_201201-201203_DART20-S0_PRECT.nc
[run] Jan2012 daily fc GPM 201201-201203 DART40-S0 PRECT
[reduce] Jan2012 GPM DART40-S0 PRECT: metrics=['RMSE', 'TCC']
         ensmean map vars: ['TCC_ensmean', 'RMSE_ensmean']


/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
